# Lesson 7.3 — Retry Limits and State Across Attempts

A counter across attempts and a budget guarantee termination: transient faults recover, persistent faults escalate, deterministic faults skip at once.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, recover, RECOVERY_POLICY)
W = GreenhouseWorld.demo_row(n=6, seed=1)
ALL_IDS = [f.fid for f in W.fruit]
TGT = understand(model_perception(W, rng=np.random.default_rng(7)), W)[1]
def kick(t, j): return 60.0 if (j == 0 and t > 0.3) else 0.0
checks = []


### Exit 1 — transient fault recovers within budget

In [2]:
r_t = recover(W, W.q.copy(), disturbance=lambda a: kick if a == 0 else None, max_attempts=3)
print('transient  -> success=%s recovered=%s n=%d' % (r_t['success'], r_t['recovered'], r_t['n_attempts']))
checks.append(r_t['success'] and r_t['recovered'] and r_t['n_attempts'] == 2)


transient  -> success=True recovered=True n=2


### Exit 2 — persistent retryable fault escalates when the budget is spent

In [3]:
r_p = recover(W, W.q.copy(), disturbance=lambda a: kick, max_attempts=3)
print('persistent -> success=%s escalated=%s n=%d reason=%s' % (r_p['success'], r_p['escalated'], r_p['n_attempts'], r_p['escalation']))
checks.append((not r_p['success']) and r_p['escalated'] and r_p['n_attempts'] == 3)
checks.append(r_p['escalation'] == 'retry-budget-exhausted')
# budget controls the count
r_p2 = recover(W, W.q.copy(), disturbance=lambda a: kick, max_attempts=5)
print('persistent (N=5) -> n=%d' % r_p2['n_attempts'])
checks.append(r_p2['n_attempts'] == 5)


persistent -> success=False escalated=True n=3 reason=retry-budget-exhausted


persistent (N=5) -> n=5


### Exit 3 — deterministic fault escalates immediately (no loop)

In [4]:
TGT = understand(model_perception(W, rng=np.random.default_rng(7)), W)[1]
r_d = recover(W, W.q.copy(), obstacle=(TGT['xy'], 0.25), max_attempts=3)
print('deterministic -> escalated=%s n=%d event=%s reason=%s' % (r_d['escalated'], r_d['n_attempts'], r_d['final_event'], r_d['escalation']))
checks.append(r_d['escalated'] and r_d['n_attempts'] == 1 and r_d['escalation'] == 'skip-target')
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


deterministic -> escalated=True n=1 event=PLAN_INVALID reason=skip-target
5/5 checks passed.
All checks passed.
